# GENOMIC INFLATION & QQ PLOTS 
 
Quantile–quantile plots and genomic inflation values for summary statistics from set-based tests using SAIGE-GENE+ are calculated for each maximum minor allele frequency threshold and test type, including burden, SKAT, and SKAT-O.
- Table 1 : Genomic inflation 
- Plot 1 : Quantile-quantile plot 

In [ ]:
# Install packages 
if (!requireNamespace("flextable", quietly = TRUE)) {
  install.packages("flextable")
}

if (!requireNamespace("officer", quietly = TRUE)) {
  install.packages("officer")
}

if (!requireNamespace("janitor", quietly = TRUE)) {
  install.packages("janitor")
}

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("biomaRt")

install.packages("xlsx")

install.packages("paletteer")

In [ ]:
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(purrr)
# Install these packages 
library(janitor)
library(flextable)
library(officer)
library(biomaRt)
library(patchwork)
library(xlsx)

In [ ]:
# Connect to Ensembl
mart <- useEnsembl(
    biomart = "genes",
    dataset = "hsapiens_gene_ensembl",
    mirror = "useast"
)

In [ ]:
# Function: Formats the gene set-based rare variant assoication test results 
process_results <- function(results) { 
    results %>% 
    filter(Group != "Cauchy") %>% 
    # inner_join(gene_symbols, by = "Region") %>% 
    mutate(max_MAF = as.character(max_MAF)) %>% 
    mutate(negative_log_p_values = -log10(Pvalue)) %>% 
    # mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
    mutate(Chromosome = as.numeric(Chromosome)) %>% 
    mutate(Region = as.factor(Region)) %>% 
    mutate(Region_factor = fct_reorder(Region, Chromosome))
  }

In [ ]:
# ========================================================================================================
# High confidence annoations (part 1)
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

files <- c()
stats <- c() 

system("mkdir -p local_dir") 
phenotypes <- c("biosample_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test: ", pheno))
        print(paste0("Loading chromosome: ", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/biosample_collection_check/EUR_",
            pheno, 
            "_high_confidence_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_high_confidence_chr", chr)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        gene_info <- getBM(
          attributes = c("ensembl_gene_id", "hgnc_symbol"),
          filters = "ensembl_gene_id",
          values = unique(processed$Region),
          mart = mart
        )

        gene_info <- 
        rename(gene_info, 
           "Region" = "ensembl_gene_id", 
           "Symbol" = "hgnc_symbol" )

        processed <- full_join(gene_info, processed, by ="Region")

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(pheno,"_high_confidence")]] <- stat
}


In [ ]:
# ========================================================================================================
# Synonymous annotations (part 1)
# ========================================================================================================

library(data.table)
library(dplyr)
library(biomaRt)

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

system("mkdir -p local_dir") 

phenotypes <- c("biosample_age")

for (pheno in phenotypes){
    
    files <- list()   # reset per phenotype
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test: ", pheno))
        print(paste0("Loading chromosome: ", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/biosample_collection_check/EUR_",
            pheno, 
            "_synonymous_chr",
            chr,
            " local_dir/"
        )

        system(cmd)

        path <- paste0(
            "local_dir/EUR_",
            pheno,
            "_synonymous_chr", chr
        )

        # If multiple files exist (common with SAIGE output)
        if (dir.exists(path)) {
            file_list <- list.files(path, full.names = TRUE)
            f <- rbindlist(lapply(file_list, fread), fill = TRUE)
        } else {
            f <- fread(path)
        }
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        gene_info <- getBM(
          attributes = c("ensembl_gene_id", "hgnc_symbol"),
          filters = "ensembl_gene_id",
          values = unique(processed$Region),
          mart = mart
        )

        gene_info <- gene_info %>%
            rename(
                Region = ensembl_gene_id,
                Symbol = hgnc_symbol
            )

        processed <- full_join(gene_info, processed, by = "Region")

        files[[chr]] <- processed
    }
    
    stat <- rbindlist(files, fill = TRUE)
    stats[[paste0(pheno, "_synonymous")]] <- stat
}

In [ ]:
# ========================================================================================================
# Table 1 : Genomic inflation 
# ========================================================================================================

df_list <- stats

genomic_inflation_list <- list()

for (df_name in names(df_list)) {
    
    df <- df_list[[df_name]]  # get the current data frame
    # Transform the full dataset for plotting
    genomic_inflation <- 
    df %>%
    filter(!is.na(Pvalue) & Pvalue > 0 & Pvalue <= 1) %>%
    pivot_longer(
    cols = starts_with("Pvalue"),
    names_to = "Statistical_Test",
    names_prefix = "Pvalue_",
    values_to = "P_value",
    values_drop_na = TRUE
    ) %>%
     mutate(
         Statistical_Test = ifelse(Statistical_Test == "Pvalue", "SKAT-O", Statistical_Test)
     ) %>%
    mutate(max_MAF = case_when(
    max_MAF == "0.01"   ~ "1 x 10⁻²",
    max_MAF == "0.001"  ~ "1 x 10⁻³",
    max_MAF == "1e-04"  ~ "1 x 10⁻⁴",
    TRUE ~ max_MAF)) %>%
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(chisq = qchisq(P_value,1,lower.tail=FALSE)) %>% 
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(lambda = (median(chisq) / qchisq(0.5,1)),
           lambda_0.01 = quantile(chisq, probs = 0.99) / qchisq(0.99, df = 1)) %>% 
    distinct(Statistical_Test,max_MAF,lambda,lambda_0.01) %>% 
    rename("Statistical Test" = "Statistical_Test", 
         "Maximum MAF" = "max_MAF", 
         "Lambda" = "lambda",
         "Lambda 0.01" = "lambda_0.01") 

    genomic_inflation$Analysis <- df_name
    genomic_inflation_list[[df_name]] <- genomic_inflation

}


In [ ]:
# Combine genomic inflation data frames 
genomic_inflation_df <- do.call(rbind, genomic_inflation_list)
genomic_inflation_df <- genomic_inflation_df[, c("Analysis", "Statistical Test", 
                                                 "Maximum MAF", "Lambda", "Lambda 0.01")]
# Write Excel
file <- "genomic_inflation.xlsx"
write.xlsx(as.data.frame(genomic_inflation_df), file)

cmd <- paste0("gsutil -m cp ", 
              file, " ", 
              my_bucket,"/data/saige_step02/HC_analysis/")

system(cmd)

In [ ]:
# ========================================================================================================
# Plot 1 : Quantile-quantile plot 
# ========================================================================================================

options(repr.plot.height = 15, 
        repr.plot.width = 15)

for (df_name in names(df_list)) {
  df <- df_list[[df_name]]  # get the current data frame
# Transform the full dataset for plotting

# Transform the full dataset for plotting
plot_data <- df %>%
  filter(!is.na(Pvalue) & Pvalue > 0 & Pvalue <= 1) %>%
  mutate(max_MAF = case_when(
    max_MAF == "0.01"  ~ "1 x 10⁻²",
    max_MAF == "0.001" ~ "1 x 10⁻³",
    max_MAF == "1e-04"  ~ "1 x 10⁻⁴",
    TRUE ~ max_MAF  # keep other values unchanged
  )) %>% 
  pivot_longer(
    cols = starts_with("Pvalue"),
    names_to = "Statistical_Test",
    names_prefix = "Pvalue_",
    values_to = "P_value",
    values_drop_na = TRUE
  ) %>%
  mutate(
    Statistical_Test = ifelse(Statistical_Test == "Pvalue", "SKAT-O", Statistical_Test)
  ) %>%
  group_by(max_MAF, Statistical_Test) %>%
  arrange(P_value, .by_group = TRUE) %>%
  mutate(
    observed = -log10(P_value),
    expected = -log10(ppoints(n()))
  ) %>%
  ungroup()

# Get the top 10 rows based on smallest P_value from the plot data
top_values <- plot_data %>%
  arrange(P_value) %>%
  slice(1:30)

# Final QQ plot with labels placed at correct expected positions
plot <- ggplot(plot_data, aes(x = expected, y = observed)) +
  geom_point(size = 2, aes(color = Statistical_Test)) +
  geom_abline(slope = 1, intercept = 0, color = "black", linetype = "dashed") +
  facet_grid(rows = vars(max_MAF), cols = vars(Statistical_Test), scales = "fixed") +
  labs(
    x = "Expected -log10(p)",
    y = "Observed -log10(p)"
  ) +
  theme_minimal() + 
  scale_color_manual(values = rev(paletteer::paletteer_d("LaCroixColoR::Lemon"))) + 
  theme(
  axis.title.x = element_text(size = 22, margin = margin(t = 10)),
  axis.title.y = element_text(size = 22, margin = margin(r = 10)),
  axis.text.x = element_text(size = 20),
  axis.text.y = element_text(size = 20),
  strip.text = element_text(size = 22),
  panel.border = element_rect(colour = "black", fill = NA, linewidth = 2) 
  ) + 
  guides(size = "none", color =  "none") +
  geom_text(
    data = top_values,
    aes(label = Symbol),
    check_overlap = TRUE,
    vjust = 0.5,
    hjust = 1.1,
    size = 5
  ) 
    
  # Save plot
  ggsave(filename = paste0("qq_", df_name, "_plot.png"), 
         plot = plot, 
         width = 15, 
         height = 15,
         dpi = 300)
}
